# Setup, Load WM

In [ ]:
DEV = False

In [ ]:
# BEG_REMOVE_DEV_rqi24u
DEV = True
%load_ext autoreload
%autoreload 2
# END_REMOVE_DEV_rqi24u

In [ ]:
WHICH_EVALS = {
    "EvalCalibSphericalShell": True,
    "EvalCalibCube": True,
    "EvalCalibMagnitude": True,
    "EvalCalibOpt": True,
    "EvalPred": True,
    "EvalSampleAroundTrue": True,
    "EvalPlanFromZero": True,
    "EvalVideoPlanFromZero": True,
}
which_eval_str = "all"
log_to_file = False
seed = 0
dataset_override = ""

In [ ]:
import sys
import os

import torch
import wandb
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm
from datetime import datetime
import plotly.graph_objects as go
from matplotlib import cm
from matplotlib.colors import Normalize, to_hex

from robot_wm.inference.actor.base import RobotActionHistory, RobotObsHistory
from robot_wm.inference.actor.cost.visual_based_latents_cost import L2VisualLatentsCost
from robot_wm.inference.robot.base import RobotObs
from robot_wm.inference.task.reference_episode import ImageProprioGoal
from robot_wm.utils.config import from_config
from robot_wm.inference.task.reference_episode.h5_imagegoal_reference_episode import H5ImageGoalReferenceEpisode
from robot_wm.inference.common.deoxys_transform_utils import euler2mat, mat2quat, quat2axisangle

In [ ]:
# Parse arguments
if not DEV:
    # args:
    # --wm_config wm_config
    import sys
    from argparse import ArgumentParser
    parser = ArgumentParser()
    parser.add_argument(
        "--wm-config",
        type=str,
        default="menagerie/config/st_wm.yaml",
        help="World model config file",
    )
    parser.add_argument(
        "--which-eval",
        type=str,
        default="all",
        help="Which eval to run",
    )
    parser.add_argument(
        "--log-to-file",
        action="store_true",
        default=False,
        help="Log to file",
    )
    parser.add_argument(
        "--seed",
        type=int,
        default=0,
        help="Random Seed",
    )
    parser.add_argument(
        "--dataset",
        type=str,
        default="",
        help="Dataset to use",
    )
    # print argv
    print("Arguments: ", end="")
    print(sys.argv)
    # remove --f= from sys.argv
    filtered_argv = [arg for arg in sys.argv if not arg.startswith("--f=")]
    filtered_argv = [arg for arg in filtered_argv if not ".py" in arg]
    # parse args
    args = parser.parse_args(filtered_argv)
    wm_config = args.wm_config
    print(args)

    if args.which_eval == "all":
        pass
    else:
        WHICH_EVALS = {k: False for k in WHICH_EVALS.keys()}
        if args.which_eval in WHICH_EVALS.keys():
            WHICH_EVALS[args.which_eval] = True
        else:
            raise ValueError(f"Unknown eval {args.which_eval}. Available: {list(WHICH_EVALS.keys())}")
        which_eval_str = args.which_eval

    log_to_file = args.log_to_file

    seed = args.seed

    dataset_override = args.dataset

    del args


In [ ]:
if DEV:
    # wm_config = 'menagerie/config/identity_wm.yaml'
    # wm_config = 'menagerie/config/st_wm.yaml'
    # wm_config = 'menagerie/config/st_wm_jepa.yaml'
    # wm_config = 'menagerie/config/st_wm_dino.yaml'
    # wm_config = 'menagerie/config/dino_wm.yaml'
    # wm_config = 'menagerie/config/jepa_wm.yaml'
    # wm_config = 'menagerie/config/mido_jepa_wm.yaml'

    # wm_config = 'menagerie/config/DINOv2-ST-1B-DROID.yaml'
    # wm_config = 'menagerie/config/DINOv2-ST-1B-DROID-rollout.yaml'
    # wm_config = 'menagerie/config/DINOv2-ST-1B-DROID-2step.yaml'
    wm_config = 'menagerie/config/JEPAv2-ST-1B-DROID.yaml'
    # wm_config = 'menagerie/config/JEPAv2-ST-1B-DROID-nocrop.yaml'
    # wm_config = 'menagerie/config/JEPAv2-Artem-DROID.yaml'
    # wm_config = 'menagerie/config/JEPAv2-Mido-DROID.yaml'
wm = from_config(wm_config)

wm_name = wm_config.split('/')[-1].split('.')[0]
print(f'Loaded {wm_name} weights')


In [ ]:
log_folder = f"outputs/{datetime.now().strftime('%Y-%m-%d/%H-%M-%S')}_{wm_name}_" + which_eval_str + ("_" + dataset_override if dataset_override != "" else "") + ("_dev" if DEV else "")
if not os.path.exists(log_folder):
    os.makedirs(log_folder)

wandb_config = {
    "wm_config": wm_config,
    "wm_name": wm_name,
    "log_folder": log_folder,
    "log_to_file": log_to_file,
    "which_evals": WHICH_EVALS,
}
wandb.init(
    project="rwm_eval" + ("_dev" if DEV else ""),
    config=wandb_config,
    id=None,
    name=wm_name + '_' + which_eval_str + ("_" + dataset_override if dataset_override != "" else ""),
    entity="robot_world_models",
)


In [ ]:

if log_to_file:
    sys.stdout = open(os.path.join(log_folder, "log.txt"), "w")
    sys.stderr = open(os.path.join(log_folder, "log.txt"), "w")


In [ ]:
def h5_to_history(sample, camera="exterior_image_1_left", resize_and_crop=(180, 320)):
    obs = sample["episode_data"]["observation"]
    observation_history = RobotObsHistory(max_context=1000, freq=30)
    for i in range(len(obs["cartesian_position"])):
        if "joint_position" in obs:
            joints = obs["joint_position"][i]
        else:
            joints = np.zeros((7,))

        ee_pose = obs["cartesian_position"][i]
        if "gripper_position" in obs:
            gripper = np.array(obs["gripper_position"][i]).reshape((1,))
        else:
            gripper = np.array([0])
        ee_pose_with_gripper = np.concatenate((ee_pose, gripper))

        if isinstance(camera, str):
            cam = obs[camera][i]
        elif isinstance(camera, list):
            cameras = [obs[c][i] for c in camera]
            cam = np.concatenate(cameras, axis=0)
        # cam2 = obs["exterior_image_2_left"][i]
        # cam = np.concatenate((cam, cam2), axis=0)
        cam = np.transpose(cam, (2, 0, 1)) / 255.0 # (H, W, C) -> (C, H, W)
        # crop to 180, 320
        if resize_and_crop is not None:
            import torchvision
            import torch
            transform = torchvision.transforms.Compose([
                torchvision.transforms.Resize(resize_and_crop),
                torchvision.transforms.CenterCrop(resize_and_crop),
            ])
            cam = transform(torch.tensor(cam)).float().cpu().numpy()

        robot_obs = RobotObs(joints, ee_pose_with_gripper, cam)
        observation_history.push(robot_obs)
    
    return observation_history

In [ ]:
# empty cell

# Evals

## Calibration

In [ ]:
if WHICH_EVALS["EvalCalibSphericalShell"]:
    EVAL_NAME = "EvalCalibSphericalShell"

    # reset random seed
    torch.manual_seed(seed)

    dset_name = "mpk"
    EPISODES = []
    for i in range(60):
        for cam in ["", "cam2_", "cam3_"]:
            EPISODES.append(f"procedural_reach_{cam}v0/task_0{i:03d}")
    CAMERAS = [
        "exterior_image_1_left",
        "exterior_image_2_left",
    ]

    if dataset_override == "droid_train":
        dset_name = "droid_train"
        droid_dataset = from_config(
            "datasets/configs/datasets/droid.yaml",
            overrides=[
                "manifest=data/datasets/droid/train_paths.csv"
            ],
        )
        # sample 100 non-overlapping indexes
        random_indexes = torch.randint(0, len(droid_dataset), (100,))
        EPISODES = [str(i.item()) for i in random_indexes]
        CAMERAS = [
            "exterior_image_1_left",
            "exterior_image_2_left",
        ]

    n_pred_steps = 1
    RES = 17
    CTX = 20
    IMAGE_GOAL = True


    results = []

    eval_folder = os.path.join(log_folder, EVAL_NAME)
    if not os.path.exists(eval_folder):
        os.makedirs(eval_folder)
    for EP in tqdm(EPISODES, desc=f"{EVAL_NAME}"):
        if DEV and len(results) > 0:
            break
        for CAM in CAMERAS:
            try:
                if dset_name == "mpk":
                    reference_episode = H5ImageGoalReferenceEpisode(
                        episode_path=f"data/evaluation_tasks/{dset_name}/{EP}/episode.h5",
                        # episode_path="data/human_demonstrations/mpk/push/boxsmallpush_v0/run_0001/episode.h5",
                        huggingface=True,
                    )
                    sample = reference_episode._reference_episode
                elif dset_name == "montreal":
                    reference_episode = H5ImageGoalReferenceEpisode(
                        episode_path=f"data/datasets/{dset_name}/{EP}/episode.h5",
                        huggingface=False,
                    )
                    sample = reference_episode._reference_episode
                elif dset_name == "droid_train":
                    sample_idx = int(EP)
                    sample = droid_dataset[sample_idx]
                else:
                    raise ValueError(f"Unknown dataset {dset_name}")

                observation_history = h5_to_history(sample, camera=CAM)

                init, context = 0, CTX
                
                n = n_pred_steps * wm.input_frames_per_prediction(observation_history.freq) 

                context_obs = observation_history[init : init + context]
                future_obs = observation_history[init + context : init + context + n]  # can be used for comparison
                context_and_future_obs = observation_history[init : init + context + n]
                context_actions = context_obs.get_action_history_from_ee_deltas()
                future_actions = future_obs.get_action_history_from_ee_deltas()
                context_and_future_actions = context_and_future_obs.get_action_history_from_ee_deltas()

            except:
                continue


            encoded_context = wm.encode_history(context_obs, context_actions)
            context_and_future_latents = wm.encode_history(context_and_future_obs, context_and_future_actions)
            if IMAGE_GOAL:
                image_goal = ImageProprioGoal(future_obs[-1].image, None)
                encoded_goal = wm.encode_goal(image_goal)

            device = "cuda"
            initial_action_tensor = future_actions.actions_tensor * 1. # (1, T, A)
            initial_action_tensor = initial_action_tensor.float().to(device)
            _1, T, A = initial_action_tensor.shape

            # find main axis and ensure rotation is not along that axis
            main_axis = initial_action_tensor[:, :, :3].sum(dim=1).squeeze(0) # (3,)
            main_axis_norm = torch.norm(main_axis)
            if main_axis_norm == 0:
                print(f"WARNING: Main axis is zero vector")
                continue
                # raise ValueError("Main axis is zero vector")
            main_axis_normalized = main_axis / main_axis_norm
            main_dim = main_axis_normalized.abs().argmax().item()


            # Rotate by up to 90 deg along two axes, in total 10 increments
            angles1 = np.linspace(-np.pi / 2, np.pi / 2, RES)
            angles2 = np.linspace(-np.pi / 2, np.pi / 2, RES)

            all_action_tensors = []
            all_cost_tensors = []
            all_iter_tensors = []
            rotated_action_tensors = [[initial_action_tensor * 1. for a2 in angles2] for a1 in angles1]
            angle_magnitudes = [[0 for a2 in angles2] for a1 in angles1]
            errors = [[0 for a2 in angles2] for a1 in angles1]
            for i1, a1 in enumerate(angles1):
                for i2, a2 in enumerate(angles2):
                    vecs = initial_action_tensor[:, :, :3] # (1, T, 3)
                    # rotate vecs by a1 and a2
                    # if main axis is in x, we rotate 
                    if main_dim == 0:
                        euler_angles = np.array([0, a1, a2])
                    elif main_dim == 1:
                        euler_angles = np.array([a1, 0, a2])
                    elif main_dim == 2:
                        euler_angles = np.array([a1, a2, 0])
                    rot_mat = euler2mat(euler_angles)
                    quat = mat2quat(rot_mat)
                    axis_angle = quat2axisangle(quat)
                    rot_mat = torch.tensor(rot_mat).float().float().to(device)
                    vecs = torch.matmul(vecs.reshape(T, 3), rot_mat)
                    rotated_action_tensors[i1][i2][:, :, :3] = vecs.reshape(1, T, 3)
                    # angle magnitude
                    angle_magnitude = np.linalg.norm(axis_angle)
                    angle_magnitudes[i1][i2] = angle_magnitude

            min_error = 1e10
            best_latents = None
            for i1, a1 in enumerate(angles1):
                for i2, a2 in enumerate(angles2):
                    rotated_action_tensor = rotated_action_tensors[i1][i2]
                    action_histories = RobotActionHistory.from_tensor(
                        1000, future_actions.freq, rotated_action_tensor
                    )
                    latent_rollouts = wm.rollout(encoded_context, action_histories)
                    # cost
                    if IMAGE_GOAL:
                        final_pose_error = L2VisualLatentsCost()(encoded_goal, latent_rollouts)[:, -1].squeeze(0).detach().float().cpu().numpy() # (S, 1)
                    else:
                        latent_L2_error = L2VideoLatentsCost()(context_and_future_latents, latent_rollouts) # (S, T)
                        final_pose_error = latent_L2_error[0, -1].detach().float().cpu().numpy()
                    errors[i1][i2] = final_pose_error
                    # store latents if cost is lower than min_error
                    if final_pose_error < min_error:
                        min_error = final_pose_error
                        best_latents = latent_rollouts

            # lowest cost
            errors = np.array(errors)
            min_cost = np.min(errors)
            min_cost_idx = np.unravel_index(np.argmin(errors), errors.shape)
            min_cost_a1 = angles1[min_cost_idx[0]]
            min_cost_a2 = angles2[min_cost_idx[1]]
            min_cost_angle_error = angle_magnitudes[min_cost_idx[0]][min_cost_idx[1]]

            if DEV:
                print(f"Angle error in degrees: {min_cost_angle_error * 180 / np.pi:.2f}")

            fig, ax = plt.subplots(2, 1)
            ax2, ax1 = ax
            # x axis should be angle 1
            # y axis should be angle 2
            imgrid = ax1.imshow(errors.T, cmap='hot', interpolation='nearest', origin='lower')
            plt.xticks(range(len(angles1)), [f"{a1:.2f}" for a1 in angles1])
            plt.yticks(range(len(angles2)), [f"{a2:.2f}" for a2 in angles2])
            # Add an arrow from (0, 0) to (a1, a2)
            if min_cost_a1 == 0 and min_cost_a2 == 0:
                # draw a green circle in the middle with no face
                ax1.scatter(RES//2, RES//2, s=100, facecolors='none', edgecolors='green')
            else:
                ax1.arrow(RES//2, RES//2, min_cost_idx[0] - RES//2, min_cost_idx[1] - RES//2, head_width=0.5, head_length=0.5, fc='green', ec='green')
            plt.colorbar(imgrid)
            plt.suptitle(f"{wm_name} - {dset_name} - {EP}")
            main_dir_str = f"[{main_axis_normalized[0].item():.2f}, {main_axis_normalized[1].item():.2f}, {main_axis_normalized[2].item():.2f}]"
            ax1.set_title(f"dir {main_dir_str} - CTX {CTX} - {n_pred_steps} steps - err: {min_cost_angle_error:.2f}")
            # Add a subplot below
            first_image = context_and_future_obs[0].image # (3, H, W)
            last_image = context_and_future_obs[-1].image # (3, H, W)
            mix_image = first_image * 0.5 + last_image * 0.5
            # convert to uint8
            mix_image = (mix_image * 255).astype(np.uint8)
            # imshow
            ax2.imshow(mix_image.transpose(1, 2, 0))
            # no ticks
            ax2.set_xticks([])
            ax2.set_yticks([])
            # plt.show()
            # savefig to log folder
            png_path = os.path.join(eval_folder, f"angleerror_{dset_name.replace('/', '-')}_{EP.replace('/', '-')}_nsteps{n_pred_steps}_CAM{CAM}_CTX{CTX}.png")
            fig.savefig(png_path)

            # show best latents
            VIDEO = False
            if VIDEO:
                pred_imgs = wm.decode_latents(best_latents)
                context_and_future_decoded = wm.decode_latents(context_and_future_latents)
                mp4_filepath = os.path.join(eval_folder, f"angleerror_{dset_name.replace('/', '-')}_EP{EP.replace('/', '-')}_nsteps{n_pred_steps}_CAM{CAM}_CTX{CTX}.mp4")
                RobotObsHistory.store_side_by_side([context_and_future_decoded, pred_imgs], filepath=mp4_filepath)

            result = {
                "calibration_error_rad": min_cost_angle_error,
                "context_length": context,
                "episode": EP,
                "camera": CAM,
                "pred_length": n,
                "pred_length_in_steps": n_pred_steps,
                "dataset": dset_name,
                "wm_name": wm_name,
            }
            results.append(result)

            # log to wandb
            wandb.log({
                f"{EVAL_NAME}/metrics/calibration_error_rad": min_cost_angle_error,
                f"{EVAL_NAME}/context_length": context,
                f"{EVAL_NAME}/episode": EP,
                f"{EVAL_NAME}/camera": CAM,
                f"{EVAL_NAME}/pred_length": n,
                f"{EVAL_NAME}/pred_length_in_steps": n_pred_steps,
                f"{EVAL_NAME}/dataset": dset_name,
                f"{EVAL_NAME}/wm_name": wm_name,
                f"{EVAL_NAME}/sample_idx": len(results),
            })
            # video
            if VIDEO:
                wandb.log({
                    f"{EVAL_NAME}/videos/{dset_name.replace('/', '-')}_EP{EP.replace('/', '-')}_nsteps{n_pred_steps}_CAM{CAM}_CTX{CTX}": wandb.Video(mp4_filepath, fps=pred_imgs.freq, format="mp4"),
                })
            # png figure
            wandb.log({
                f"{EVAL_NAME}/figures/{dset_name.replace('/', '-')}_EP{EP.replace('/', '-')}_nsteps{n_pred_steps}_CAM{CAM}_CTX{CTX}": wandb.Image(png_path),
            })

            if DEV:
                plt.show()
                # raise ValueError

    # log as wandb table
    table = wandb.Table(columns=[key for key in results[0].keys()], data=[list(result.values()) for result in results])
    wandb.log({
        f"{EVAL_NAME}/table": table,
    })
    wandb.log({
        f"{EVAL_NAME}/metrics/camera_dependent_calibration_error": wandb.plot.bar(
            table, "camera", "calibration_error_rad", title="Calibration error by camera")
    })

    # mean calibration error
    mean_calibration_error = np.mean([result["calibration_error_rad"] for result in results])
    print(f"Mean calibration error: {mean_calibration_error:.2f} rad")
    # log to wandb
    wandb.log({
        f"{EVAL_NAME}/metrics/mean_calibration_error_rad": mean_calibration_error,
    })



## Calibration (Cube)

In [ ]:
if WHICH_EVALS["EvalCalibCube"]:
    EVAL_NAME = "EvalCalibCube"

    # reset random seed
    torch.manual_seed(seed)

    dset_name = "mpk"
    EPISODES = []
    for i in range(60):
        for cam in ["", "cam2_", "cam3_"]:
            EPISODES.append(f"procedural_reach_{cam}v0/task_0{i:03d}")
    CAMERAS = [
        "exterior_image_1_left",
        "exterior_image_2_left",
    ]

    if dataset_override == "droid_train":
        dset_name = "droid_train"
        droid_dataset = from_config(
            "datasets/configs/datasets/droid.yaml",
            overrides=[
                "manifest=data/datasets/droid/train_paths.csv"
            ],
        )
        # sample 100 non-overlapping indexes
        random_indexes = torch.randint(0, len(droid_dataset), (100,))
        EPISODES = [str(i.item()) for i in random_indexes]
        CAMERAS = [
            "exterior_image_1_left",
            "exterior_image_2_left",
        ]

    n_pred_steps = 1
    RES = 9
    CTX = 20
    IMAGE_GOAL = True


    results = []

    eval_folder = os.path.join(log_folder, EVAL_NAME)
    if not os.path.exists(eval_folder):
        os.makedirs(eval_folder)
    for EP in tqdm(EPISODES, desc=f"{EVAL_NAME}"):
        if DEV and len(results) > 0:
            break
        for CAM in CAMERAS:
            try:
                if dset_name == "mpk":
                    reference_episode = H5ImageGoalReferenceEpisode(
                        episode_path=f"data/evaluation_tasks/{dset_name}/{EP}/episode.h5",
                        # episode_path="data/human_demonstrations/mpk/push/boxsmallpush_v0/run_0001/episode.h5",
                        huggingface=True,
                    )
                    sample = reference_episode._reference_episode
                elif dset_name == "montreal":
                    reference_episode = H5ImageGoalReferenceEpisode(
                        episode_path=f"data/datasets/{dset_name}/{EP}/episode.h5",
                        huggingface=False,
                    )
                    sample = reference_episode._reference_episode
                elif dset_name == "droid_train":
                    sample_idx = int(EP)
                    sample = droid_dataset[sample_idx]
                else:
                    raise ValueError(f"Unknown dataset {dset_name}")

                observation_history = h5_to_history(sample, camera=CAM)

                init, context = 0, CTX
                
                n = n_pred_steps * wm.input_frames_per_prediction(observation_history.freq) 

                context_obs = observation_history[init : init + context]
                future_obs = observation_history[init + context : init + context + n]  # can be used for comparison
                context_and_future_obs = observation_history[init : init + context + n]
                context_actions = context_obs.get_action_history_from_ee_deltas()
                future_actions = future_obs.get_action_history_from_ee_deltas()
                context_and_future_actions = context_and_future_obs.get_action_history_from_ee_deltas()

            except:
                continue


            encoded_context = wm.encode_history(context_obs, context_actions)
            context_and_future_latents = wm.encode_history(context_and_future_obs, context_and_future_actions)
            if IMAGE_GOAL:
                image_goal = ImageProprioGoal(future_obs[-1].image, None)
                encoded_goal = wm.encode_goal(image_goal)

            device = "cuda"
            initial_action_tensor = future_actions.actions_tensor * 1. # (1, T, A)
            initial_action_tensor = initial_action_tensor.float().to(device)
            _1, T, A = initial_action_tensor.shape

            # find main axis and ensure rotation is not along that axis
            main_axis = initial_action_tensor[:, :, :3].sum(dim=1).squeeze(0) # (3,)
            main_axis_norm = torch.norm(main_axis)
            if main_axis_norm == 0:
                print(f"WARNING: Main axis is zero vector")
                continue
                # raise ValueError("Main axis is zero vector")
            main_axis_normalized = main_axis / main_axis_norm
            main_dim = main_axis_normalized.abs().argmax().item()


            # Perturb true vector
            MAX_D = 0.02
            dxs = np.linspace(-MAX_D, MAX_D, RES)
            dys = np.linspace(-MAX_D, MAX_D, RES)
            dzs = np.linspace(-MAX_D, MAX_D, RES)

            all_action_tensors = []
            all_cost_tensors = []
            all_iter_tensors = []
            perturbed_action_tensors = [[[initial_action_tensor * 1. for dz in dzs] for dy in dys] for dx in dxs]
            perturb_magnitudes = [[[0 for dz in dzs] for dy in dys] for dx in dxs]
            errors = [[[0 for dz in dzs] for dy in dys] for dx in dxs]
            for i1, dx in enumerate(dxs):
                for i2, dy in enumerate(dys):
                    for i3, dz in enumerate(dzs):
                        vecs = initial_action_tensor[:, :, :3] # (1, T, 3)
                        # rotate vecs by dx and dy
                        # if main axis is in x, we rotate 
                        perturbation = torch.tensor([dx, dy, dz]).unsqueeze(0).to(initial_action_tensor) # (1, 3)
                        vecs = vecs.reshape(T, 3) + perturbation
                        perturbed_action_tensors[i1][i2][i3][:, :, :3] = vecs.reshape(1, T, 3)
                        # perturbation magnitude
                        perturb_magnitude = np.linalg.norm(perturbation.cpu().numpy())
                        perturb_magnitudes[i1][i2][i3] = perturb_magnitude

            min_error = 1e10
            best_latents = None
            for i1, dx in enumerate(dxs):
                for i2, dy in enumerate(dys):
                    for i3, dz in enumerate(dzs):
                        perturbed_action_tensor = perturbed_action_tensors[i1][i2][i3]
                        action_histories = RobotActionHistory.from_tensor(
                            1000, future_actions.freq, perturbed_action_tensor
                        )
                        latent_rollouts = wm.rollout(encoded_context, action_histories)
                        # cost
                        if IMAGE_GOAL:
                            final_pose_error = L2VisualLatentsCost()(encoded_goal, latent_rollouts)[:, -1].squeeze(0).detach().float().cpu().numpy() # (S, 1)
                        else:
                            latent_L2_error = L2VideoLatentsCost()(context_and_future_latents, latent_rollouts) # (S, T)
                            final_pose_error = latent_L2_error[0, -1].detach().float().cpu().numpy()
                        errors[i1][i2][i3] = final_pose_error
                        # store latents if cost is lower than min_error
                        if final_pose_error < min_error:
                            min_error = final_pose_error
                            best_latents = latent_rollouts

            # lowest cost
            errors = np.array(errors)
            min_cost = np.min(errors)
            min_cost_idx = np.unravel_index(np.argmin(errors), errors.shape)
            min_cost_dx = dxs[min_cost_idx[0]]
            min_cost_dy = dys[min_cost_idx[1]]
            min_cost_dz = dzs[min_cost_idx[2]]
            min_cost_perturb_error = perturb_magnitudes[min_cost_idx[0]][min_cost_idx[1]][min_cost_idx[2]]

            if DEV:
                print(f"Perturbation error in meters: {min_cost_perturb_error:.2f}")

            from matplotlib import gridspec
            fig = plt.figure(figsize=(10, 6))
            gs = gridspec.GridSpec(2, 3, height_ratios=[1, 1])

            # Top subplot spanning all columns
            ax0 = fig.add_subplot(gs[0, :])

            # Bottom row: three subplots
            ax1 = fig.add_subplot(gs[1, 0])
            ax2 = fig.add_subplot(gs[1, 1])
            ax3 = fig.add_subplot(gs[1, 2])
            # top view
            imgrid = ax1.imshow(errors[:, :, RES//2].T, cmap='hot', interpolation='nearest', origin='lower')
            plt.sca(ax1)
            plt.xticks(range(len(dxs)), [f"{dx:.2f}" for dx in dxs])
            plt.yticks(range(len(dys)), [f"{dy:.2f}" for dy in dys])
            plt.colorbar(imgrid)
            ax1.set_xlabel("dx")
            ax1.set_ylabel("dy")
            # Add an arrow from (0, 0) to (dx, dy)
            if min_cost_dx == 0 and min_cost_dy == 0 and min_cost_dz == 0:
                # draw a green circle in the middle with no face
                ax1.scatter(RES//2, RES//2, s=100, facecolors='none', edgecolors='green')
            else:
                ax1.arrow(RES//2, RES//2, min_cost_idx[0] - RES//2, min_cost_idx[1] - RES//2, head_width=0.5, head_length=0.5, fc='green', ec='green')
            # side view
            imgrid = ax2.imshow(errors[:, RES//2, :].T, cmap='hot', interpolation='nearest', origin='lower')
            plt.sca(ax2)
            plt.xticks(range(len(dxs)), [f"{dx:.2f}" for dx in dxs])
            plt.yticks(range(len(dzs)), [f"{dz:.2f}" for dz in dzs])
            plt.colorbar(imgrid)
            ax2.set_xlabel("dx")
            ax2.set_ylabel("dz")
            # Add an arrow from (0, 0) to (dx, dy)
            if min_cost_dx == 0 and min_cost_dy == 0 and min_cost_dz == 0:
                # draw a green circle in the middle with no face
                ax2.scatter(RES//2, RES//2, s=100, facecolors='none', edgecolors='green')
            else:
                ax2.arrow(RES//2, RES//2, min_cost_idx[0] - RES//2, min_cost_idx[2] - RES//2, head_width=0.5, head_length=0.5, fc='green', ec='green')
            # front view
            imgrid = ax3.imshow(errors[RES//2, :, :].T, cmap='hot', interpolation='nearest', origin='lower')
            plt.sca(ax3)
            plt.xticks(range(len(dys)), [f"{dy:.2f}" for dy in dys])
            plt.yticks(range(len(dzs)), [f"{dz:.2f}" for dz in dzs])
            ax3.set_xlabel("dy")
            ax3.set_ylabel("dz")
            # Add an arrow from (0, 0) to (dx, dy)
            if min_cost_dx == 0 and min_cost_dy == 0 and min_cost_dz == 0:
                # draw a green circle in the middle with no face
                ax3.scatter(RES//2, RES//2, s=100, facecolors='none', edgecolors='green')
            else:
                ax3.arrow(RES//2, RES//2, min_cost_idx[1] - RES//2, min_cost_idx[2] - RES//2, head_width=0.5, head_length=0.5, fc='green', ec='green')
            plt.colorbar(imgrid)
            plt.suptitle(f"{wm_name} - {dset_name} - {EP}")
            main_dir_str = f"[{main_axis[0].item():.3f}, {main_axis[1].item():.3f}, {main_axis[2].item():.3f}]"
            best_dir_str = f"[{main_axis[0].item()+min_cost_dx:.3f}, {main_axis[1].item()+min_cost_dy:.3f}, {main_axis[2].item()+min_cost_dz:.3f}]"
            ax0.set_title(f"dir {main_dir_str} - best {best_dir_str} - CTX {CTX} - {n_pred_steps} steps - err: {min_cost_perturb_error:.3f}")
            # Add a subplot below
            first_image = context_and_future_obs[0].image # (3, H, W)
            last_image = context_and_future_obs[-1].image # (3, H, W)
            mix_image = first_image * 0.5 + last_image * 0.5
            # convert to uint8
            mix_image = (mix_image * 255).astype(np.uint8)
            # imshow
            ax0.imshow(mix_image.transpose(1, 2, 0))
            # no ticks
            ax0.set_xticks([])
            ax0.set_yticks([])
            # plt.show()
            # savefig to log folder
            png_path = os.path.join(eval_folder, f"perturberror_{dset_name.replace('/', '-')}_{EP.replace('/', '-')}_nsteps{n_pred_steps}_CAM{CAM}_CTX{CTX}.png")
            fig.savefig(png_path)

            # show best latents
            VIDEO = False
            if VIDEO:
                pred_imgs = wm.decode_latents(best_latents)
                context_and_future_decoded = wm.decode_latents(context_and_future_latents)
                mp4_filepath = os.path.join(eval_folder, f"perturberror_{dset_name.replace('/', '-')}_EP{EP.replace('/', '-')}_nsteps{n_pred_steps}_CAM{CAM}_CTX{CTX}.mp4")
                RobotObsHistory.store_side_by_side([context_and_future_decoded, pred_imgs], filepath=mp4_filepath)

            result = {
                "calibration_error_m": min_cost_perturb_error,
                "context_length": context,
                "episode": EP,
                "camera": CAM,
                "pred_length": n,
                "pred_length_in_steps": n_pred_steps,
                "dataset": dset_name,
                "wm_name": wm_name,
            }
            results.append(result)

            # log to wandb
            wandb.log({
                f"{EVAL_NAME}/metrics/calibration_error_m": min_cost_perturb_error,
                f"{EVAL_NAME}/context_length": context,
                f"{EVAL_NAME}/episode": EP,
                f"{EVAL_NAME}/camera": CAM,
                f"{EVAL_NAME}/pred_length": n,
                f"{EVAL_NAME}/pred_length_in_steps": n_pred_steps,
                f"{EVAL_NAME}/dataset": dset_name,
                f"{EVAL_NAME}/wm_name": wm_name,
                f"{EVAL_NAME}/sample_idx": len(results),
            })
            # video
            if VIDEO:
                wandb.log({
                    f"{EVAL_NAME}/videos/{dset_name.replace('/', '-')}_EP{EP.replace('/', '-')}_nsteps{n_pred_steps}_CAM{CAM}_CTX{CTX}": wandb.Video(mp4_filepath, fps=pred_imgs.freq, format="mp4"),
                })
            # png figure
            wandb.log({
                f"{EVAL_NAME}/figures/{dset_name.replace('/', '-')}_EP{EP.replace('/', '-')}_nsteps{n_pred_steps}_CAM{CAM}_CTX{CTX}": wandb.Image(png_path),
            })

            if DEV:
                plt.show()
                # raise ValueError

    # log as wandb table
    table = wandb.Table(columns=[key for key in results[0].keys()], data=[list(result.values()) for result in results])
    wandb.log({
        f"{EVAL_NAME}/table": table,
    })
    wandb.log({
        f"{EVAL_NAME}/metrics/camera_dependent_calibration_error": wandb.plot.bar(
            table, "camera", "calibration_error_m", title="Calibration error by camera")
    })

    # mean calibration error
    mean_calibration_error = np.mean([result["calibration_error_m"] for result in results])
    print(f"Mean calibration error: {mean_calibration_error:.3f} m")
    # log to wandb
    wandb.log({
        f"{EVAL_NAME}/metrics/mean_calibration_error_m": mean_calibration_error,
    })



## Calibration (Norm)

In [ ]:
if WHICH_EVALS["EvalCalibMagnitude"]:
    EVAL_NAME = "EvalCalibMagnitude"

    # reset random seed
    torch.manual_seed(seed)

    dset_name = "mpk"
    EPISODES = []
    for i in range(60):
        for cam in ["", "cam2_", "cam3_"]:
            EPISODES.append(f"procedural_reach_{cam}v0/task_0{i:03d}")
    CAMERAS = [
        "exterior_image_1_left",
        "exterior_image_2_left",
    ]

    if dataset_override == "droid_train":
        dset_name = "droid_train"
        droid_dataset = from_config(
            "datasets/configs/datasets/droid.yaml",
            overrides=[
                "manifest=data/datasets/droid/train_paths.csv"
            ],
        )
        # sample 100 non-overlapping indexes
        random_indexes = torch.randint(0, len(droid_dataset), (100,))
        EPISODES = [str(i.item()) for i in random_indexes]
        CAMERAS = [
            "exterior_image_1_left",
            "exterior_image_2_left",
        ]

    n_pred_steps = 1
    RES = 17
    CTX = 20
    IMAGE_GOAL = True


    results = []

    eval_folder = os.path.join(log_folder, EVAL_NAME)
    if not os.path.exists(eval_folder):
        os.makedirs(eval_folder)
    for EP in tqdm(EPISODES, desc=f"{EVAL_NAME}"):
        if DEV and len(results) > 0:
            break
        for CAM in CAMERAS:
            try:
                if dset_name == "mpk":
                    reference_episode = H5ImageGoalReferenceEpisode(
                        episode_path=f"data/evaluation_tasks/{dset_name}/{EP}/episode.h5",
                        # episode_path="data/human_demonstrations/mpk/push/boxsmallpush_v0/run_0001/episode.h5",
                        huggingface=True,
                    )
                    sample = reference_episode._reference_episode
                elif dset_name == "montreal":
                    reference_episode = H5ImageGoalReferenceEpisode(
                        episode_path=f"data/datasets/{dset_name}/{EP}/episode.h5",
                        huggingface=False,
                    )
                    sample = reference_episode._reference_episode
                elif dset_name == "droid_train":
                    sample_idx = int(EP)
                    sample = droid_dataset[sample_idx]
                else:
                    raise ValueError(f"Unknown dataset {dset_name}")

                observation_history = h5_to_history(sample, camera=CAM)

                init, context = 0, CTX
                
                n = n_pred_steps * wm.input_frames_per_prediction(observation_history.freq) 

                context_obs = observation_history[init : init + context]
                future_obs = observation_history[init + context : init + context + n]  # can be used for comparison
                context_and_future_obs = observation_history[init : init + context + n]
                context_actions = context_obs.get_action_history_from_ee_deltas()
                future_actions = future_obs.get_action_history_from_ee_deltas()
                context_and_future_actions = context_and_future_obs.get_action_history_from_ee_deltas()

            except:
                print(f"Warning: skipping episode {EP} - camera {CAM}")
                continue


            encoded_context = wm.encode_history(context_obs, context_actions)
            context_and_future_latents = wm.encode_history(context_and_future_obs, context_and_future_actions)
            if IMAGE_GOAL:
                image_goal = ImageProprioGoal(future_obs[-1].image, None)
                encoded_goal = wm.encode_goal(image_goal)

            device = "cuda"
            initial_action_tensor = future_actions.actions_tensor * 1. # (1, T, A)
            initial_action_tensor = initial_action_tensor.float().to(device)
            _1, T, A = initial_action_tensor.shape

            # find main axis and ensure rotation is not along that axis
            main_axis = initial_action_tensor[:, :, :3].sum(dim=1).squeeze(0) # (3,)
            main_axis_norm = torch.norm(main_axis)
            if main_axis_norm == 0:
                print(f"WARNING: Main axis is zero vector")
                continue
                # raise ValueError("Main axis is zero vector")
            main_axis_normalized = main_axis / main_axis_norm
            main_dim = main_axis_normalized.abs().argmax().item()


            # Perturb true vector
            length_factors = np.linspace(0, 2, RES)

            all_action_tensors = []
            all_cost_tensors = []
            all_iter_tensors = []
            perturbed_action_tensors = [initial_action_tensor * 1. for l in length_factors]
            errors = [0 for l in length_factors]
            for i1, l in enumerate(length_factors):
                vecs = initial_action_tensor[:, :, :3] # (1, T, 3)
                perturbed_action_tensors[i1][:, :, :3] = vecs * l

            min_error = 1e10
            best_latents = None
            for i1, l in enumerate(length_factors):
                perturbed_action_tensor = perturbed_action_tensors[i1]
                action_histories = RobotActionHistory.from_tensor(
                    1000, future_actions.freq, perturbed_action_tensor
                )
                latent_rollouts = wm.rollout(encoded_context, action_histories)
                # cost
                if IMAGE_GOAL:
                    final_pose_error = L2VisualLatentsCost()(encoded_goal, latent_rollouts)[:, -1].squeeze(0).detach().float().cpu().numpy() # (S, 1)
                else:
                    latent_L2_error = L2VideoLatentsCost()(context_and_future_latents, latent_rollouts) # (S, T)
                    final_pose_error = latent_L2_error[0, -1].detach().float().cpu().numpy()
                errors[i1] = final_pose_error
                # store latents if cost is lower than min_error
                if final_pose_error < min_error:
                    min_error = final_pose_error
                    best_latents = latent_rollouts

            # lowest cost
            errors = np.array(errors)
            min_cost = np.min(errors)
            min_cost_idx = np.argmin(errors)
            min_cost_l = length_factors[min_cost_idx]
            min_cost_perturb_error = abs(min_cost_l - 1) * 100.

            if DEV:
                print(f"Perturbation error in %: {min_cost_perturb_error:.2f}")

            from matplotlib import gridspec
            fig = plt.figure(figsize=(10, 6))
            ax0, ax1 = fig.subplots(2, 1)
            # top view
            imgrid = ax1.plot(length_factors, errors)
            ax1.axvline(x=1, color='black', linestyle='--')
            if min_cost_l == 1:
                ax1.axvline(x=min_cost_l, color='green', linestyle='--')
            else:
                ax1.axvline(x=min_cost_l, color='red', linestyle='--')
            plt.sca(ax1)
            ax1.set_xlabel("length factor")
            ax1.set_ylabel("cost")
            # title
            plt.suptitle(f"{wm_name} - {dset_name} - {EP}")
            main_dir_str = f"[{main_axis[0].item():.3f}, {main_axis[1].item():.3f}, {main_axis[2].item():.3f}]"
            best_dir_str = f"[{main_axis[0].item()*min_cost_l:.3f}, {main_axis[1].item()*min_cost_l:.3f}, {main_axis[2].item()*min_cost_l:.3f}]"
            ax0.set_title(f"dir {main_dir_str} - best {best_dir_str} - CTX {CTX} - {n_pred_steps} steps - {RES} res - err: {min_cost_perturb_error:.0f}")
            # Add a subplot below
            first_image = context_and_future_obs[0].image # (3, H, W)
            last_image = context_and_future_obs[-1].image # (3, H, W)
            mix_image = first_image * 0.5 + last_image * 0.5
            # convert to uint8
            mix_image = (mix_image * 255).astype(np.uint8)
            # imshow
            ax0.imshow(mix_image.transpose(1, 2, 0))
            # no ticks
            ax0.set_xticks([])
            ax0.set_yticks([])
            # plt.show()
            # savefig to log folder
            png_path = os.path.join(eval_folder, f"perturberror_{dset_name.replace('/', '-')}_{EP.replace('/', '-')}_nsteps{n_pred_steps}_CAM{CAM}_CTX{CTX}.png")
            fig.savefig(png_path)

            # show best latents
            VIDEO = False
            if VIDEO:
                pred_imgs = wm.decode_latents(best_latents)
                context_and_future_decoded = wm.decode_latents(context_and_future_latents)
                mp4_filepath = os.path.join(eval_folder, f"perturberror_{dset_name.replace('/', '-')}_EP{EP.replace('/', '-')}_nsteps{n_pred_steps}_CAM{CAM}_CTX{CTX}.mp4")
                RobotObsHistory.store_side_by_side([context_and_future_decoded, pred_imgs], filepath=mp4_filepath)

            result = {
                "calibration_error_%": min_cost_perturb_error,
                "context_length": context,
                "episode": EP,
                "camera": CAM,
                "pred_length": n,
                "pred_length_in_steps": n_pred_steps,
                "dataset": dset_name,
                "wm_name": wm_name,
            }
            results.append(result)

            # log to wandb
            wandb.log({
                f"{EVAL_NAME}/metrics/calibration_error_%": min_cost_perturb_error,
                f"{EVAL_NAME}/context_length": context,
                f"{EVAL_NAME}/episode": EP,
                f"{EVAL_NAME}/camera": CAM,
                f"{EVAL_NAME}/pred_length": n,
                f"{EVAL_NAME}/pred_length_in_steps": n_pred_steps,
                f"{EVAL_NAME}/dataset": dset_name,
                f"{EVAL_NAME}/wm_name": wm_name,
                f"{EVAL_NAME}/sample_idx": len(results),
            })
            # video
            if VIDEO:
                wandb.log({
                    f"{EVAL_NAME}/videos/{dset_name.replace('/', '-')}_EP{EP.replace('/', '-')}_nsteps{n_pred_steps}_CAM{CAM}_CTX{CTX}": wandb.Video(mp4_filepath, fps=pred_imgs.freq, format="mp4"),
                })
            # png figure
            wandb.log({
                f"{EVAL_NAME}/figures/{dset_name.replace('/', '-')}_EP{EP.replace('/', '-')}_nsteps{n_pred_steps}_CAM{CAM}_CTX{CTX}": wandb.Image(png_path),
            })

            if DEV:
                plt.show()
                # raise ValueError

    # log as wandb table
    table = wandb.Table(columns=[key for key in results[0].keys()], data=[list(result.values()) for result in results])
    wandb.log({
        f"{EVAL_NAME}/table": table,
    })
    wandb.log({
        f"{EVAL_NAME}/metrics/camera_dependent_calibration_error": wandb.plot.bar(
            table, "camera", "calibration_error_%", title="Calibration error by camera")
    })

    # mean calibration error
    mean_calibration_error = np.mean([result["calibration_error_%"] for result in results])
    print(f"Mean calibration error: {mean_calibration_error:.0f} %")
    # log to wandb
    wandb.log({
        f"{EVAL_NAME}/metrics/mean_calibration_error_%": mean_calibration_error,
    })




## Calibration (Optimized)

In [ ]:
if WHICH_EVALS["EvalCalibOpt"]:
    EVAL_NAME = "EvalCalibOpt"

    # reset random seed
    torch.manual_seed(seed)

    dset_name = "mpk"
    EPISODES = []
    for i in range(60):
        for cam in ["", "cam2_", "cam3_"]:
            EPISODES.append(f"procedural_reach_{cam}v0/task_0{i:03d}")
    CAMERAS = [
        "exterior_image_1_left",
        "exterior_image_2_left",
    ]

    if dataset_override == "droid_train":
        dset_name = "droid_train"
        droid_dataset = from_config(
            "datasets/configs/datasets/droid.yaml",
            overrides=[
                "manifest=data/datasets/droid/train_paths.csv"
            ],
        )
        # sample 100 non-overlapping indexes
        random_indexes = torch.randint(0, len(droid_dataset), (100,))
        EPISODES = [str(i.item()) for i in random_indexes]
        CAMERAS = [
            "exterior_image_1_left",
            "exterior_image_2_left",
        ]

    n_pred_steps = 1
    CTX = 20
    B = 10
    ITER = 200
    IMAGE_GOAL = True
    ANNEALING = True
    ADD_ZERO_SAMPLE = True


    results = []

    # reset random seed
    torch.manual_seed(seed)

    eval_folder = os.path.join(log_folder, EVAL_NAME)
    if not os.path.exists(eval_folder):
        os.makedirs(eval_folder)
    for EP in tqdm(EPISODES, desc=f"{EVAL_NAME}"):
        if DEV and len(results) > 0:
            break
        for CAM in CAMERAS:
            try:
                if dset_name == "mpk":
                    reference_episode = H5ImageGoalReferenceEpisode(
                        episode_path=f"data/evaluation_tasks/{dset_name}/{EP}/episode.h5",
                        # episode_path="data/human_demonstrations/mpk/push/boxsmallpush_v0/run_0001/episode.h5",
                        huggingface=True,
                    )
                    sample = reference_episode._reference_episode
                elif dset_name == "montreal":
                    reference_episode = H5ImageGoalReferenceEpisode(
                        episode_path=f"data/datasets/{dset_name}/{EP}/episode.h5",
                        huggingface=False,
                    )
                    sample = reference_episode._reference_episode
                elif dset_name == "droid_train":
                    sample_idx = int(EP)
                    sample = droid_dataset[sample_idx]
                else:
                    raise ValueError(f"Unknown dataset {dset_name}")

                observation_history = h5_to_history(sample, camera=CAM)

                init, context = 0, CTX
                
                n = n_pred_steps * wm.input_frames_per_prediction(observation_history.freq) 

                context_obs = observation_history[init : init + context]
                future_obs = observation_history[init + context : init + context + n]  # can be used for comparison
                context_and_future_obs = observation_history[init : init + context + n]
                context_actions = context_obs.get_action_history_from_ee_deltas()
                future_actions = future_obs.get_action_history_from_ee_deltas()
                context_and_future_actions = context_and_future_obs.get_action_history_from_ee_deltas()

            except:
                continue


            encoded_context = wm.encode_history(context_obs, context_actions)
            context_and_future_latents = wm.encode_history(context_and_future_obs, context_and_future_actions)
            if IMAGE_GOAL:
                image_goal = ImageProprioGoal(future_obs[-1].image, None)
                encoded_goal = wm.encode_goal(image_goal)

            device = "cuda"
            initial_action_tensor = future_actions.actions_tensor * 1. # (1, T, A)
            initial_action_tensor = initial_action_tensor.float().to(device)
            _1, T, A = initial_action_tensor.shape

            # find main axis and ensure rotation is not along that axis
            main_axis = initial_action_tensor[:, :, :3].sum(dim=1).squeeze(0) # (3,)
            main_axis_norm = torch.norm(main_axis)
            if main_axis_norm == 0:
                print(f"WARNING: Main axis is zero vector")
                continue
                # raise ValueError("Main axis is zero vector")
            main_axis_normalized = main_axis / main_axis_norm
            main_dim = main_axis_normalized.abs().argmax().item()


            # Perturb true vector
            MAX_D = 0.02
            SIGMAS = [0.01, 0.01, 0.01, 0, 0, 0, 0]

            dxs = []
            dys = []
            dzs = []
            all_action_tensors = []
            all_cost_tensors = []
            all_iter_tensors = []
            errors = []
            iteridx = []

            min_error = 1e10
            best_latents = None
            best_action_tensor = initial_action_tensor * 1.0
            best_perturbations = torch.zeros((1, 1, A)).to(device)
            best_deltas = None
            for itr in range(ITER):
                # TODO: check that it makes sense to apply same perturb to all timesteps (1 instead of T)
                cooling_factor = 1.0
                if ANNEALING:
                    iter_01 = itr / ITER
                    cooling_factor = 1.0 - iter_01
                random_perturbations = torch.randn((B, 1, A)).to(device) * torch.tensor(SIGMAS).unsqueeze(0).unsqueeze(0).to(device) * cooling_factor # (B, 1, A)
                if ANNEALING:
                    random_perturbations += best_perturbations
                random_perturbations[:, :, 3:] = 0
                if ADD_ZERO_SAMPLE and itr == 0:
                    random_perturbations[0, :, :] = 0 # make sure one sample out of all is 0 itself
                    zero_sample_idx = len(errors)
                integrated_random_perturbations = random_perturbations.sum(axis=1) # (B, A)
                dxs.extend(integrated_random_perturbations[:, 0].detach().cpu().numpy())
                dys.extend(integrated_random_perturbations[:, 1].detach().cpu().numpy())
                dzs.extend(integrated_random_perturbations[:, 2].detach().cpu().numpy())

                perturbed_action_tensor = initial_action_tensor + random_perturbations
                action_histories = RobotActionHistory.from_tensor(
                    1000, future_actions.freq, perturbed_action_tensor
                )
                latent_rollouts = wm.rollout(encoded_context, action_histories)
                # cost
                if IMAGE_GOAL:
                    final_pose_errors = L2VisualLatentsCost()(encoded_goal, latent_rollouts)[:, -1].detach().float().cpu().numpy() # (B,)
                else:
                    latent_L2_error = L2VideoLatentsCost()(context_and_future_latents, latent_rollouts) # (B, T)
                    final_pose_errors = latent_L2_error[:, -1].detach().float().cpu().numpy() # (B,)
                errors.extend(final_pose_errors)
                iteridx.extend([itr] * B)

                # store latents if cost is lower than min_error
                batch_best_idx = np.argmin(final_pose_errors)
                batch_best_latent = latent_rollouts.pick_sample(batch_best_idx)
                batch_best_error = final_pose_errors[batch_best_idx]
                batch_best_action_tensor = perturbed_action_tensor[batch_best_idx].unsqueeze(0)
                if batch_best_error < min_error:
                    min_error = batch_best_error
                    best_latents = batch_best_latent
                    best_action_tensor = batch_best_action_tensor
                    best_deltas = integrated_random_perturbations[batch_best_idx, :3].detach().cpu().numpy()
                    best_perturbations = random_perturbations[batch_best_idx, :, :].unsqueeze(0)

            # lowest cost
            errors = np.array(errors)
            min_cost = min_error
            min_cost_dx, min_cost_dy, min_cost_dz = best_deltas
            min_cost_perturb_error = np.linalg.norm(np.array([min_cost_dx, min_cost_dy, min_cost_dz]))

            print(f"Perturbation error in meters: {min_cost_perturb_error:.2f}")

            from matplotlib import gridspec
            fig = plt.figure(figsize=(10, 6))
            gs = gridspec.GridSpec(2, 3, height_ratios=[1, 1])

            # Top subplot spanning all columns
            ax0 = fig.add_subplot(gs[0, :])

            # Bottom row: three subplots
            ax1 = fig.add_subplot(gs[1, 0])
            ax2 = fig.add_subplot(gs[1, 1])
            ax3 = fig.add_subplot(gs[1, 2])

            # top view
            # imgrid = ax1.imshow(errors[:, :, RES//2].T, cmap='hot', interpolation='nearest', origin='lower')
            scatter = ax1.scatter(dxs, dys, c=errors, marker='o', alpha=0.3)
            plt.sca(ax1)
            # plt.xticks(range(len(dxs)), [f"{dx:.2f}" for dx in dxs])
            # plt.yticks(range(len(dys)), [f"{dy:.2f}" for dy in dys])
            plt.colorbar(scatter)
            ax1.set_xlabel("dx")
            ax1.set_ylabel("dy")
            # Add an arrow from (0, 0) to (dx, dy)
            if min_cost_dx == 0 and min_cost_dy == 0 and min_cost_dz == 0:
                # draw a green circle in the middle with no face
                ax1.scatter(0, 0, s=100, facecolors='none', edgecolors='green')
            else:
                ax1.arrow(0, 0, min_cost_dx, min_cost_dy, width=0.0005,  fc='red', ec='red')
            # side view
            scatter = ax2.scatter(dxs, dzs, c=errors, marker='o', alpha=0.3)
            plt.sca(ax2)
            # plt.xticks(range(len(dxs)), [f"{dx:.2f}" for dx in dxs])
            # plt.yticks(range(len(dys)), [f"{dy:.2f}" for dy in dys])
            plt.colorbar(scatter)
            ax2.set_xlabel("dx")
            ax2.set_ylabel("dz")
            # Add an arrow from (0, 0) to (dx, dy)
            if min_cost_dx == 0 and min_cost_dy == 0 and min_cost_dz == 0:
                # draw a green circle in the middle with no face
                ax2.scatter(0, 0, s=100, facecolors='none', edgecolors='green')
            else:
                ax2.arrow(0, 0, min_cost_dx, min_cost_dz, width=0.0005, fc='red', ec='red')
            # front view
            scatter = ax3.scatter(dys, dzs, c=errors, marker='o', alpha=0.3)
            plt.sca(ax3)
            # plt.xticks(range(len(dxs)), [f"{dx:.2f}" for dx in dxs])
            # plt.yticks(range(len(dys)), [f"{dy:.2f}" for dy in dys])
            plt.colorbar(scatter)
            ax3.set_xlabel("dy")
            ax3.set_ylabel("dz")
            # Add an arrow from (0, 0) to (dx, dy)
            if min_cost_dx == 0 and min_cost_dy == 0 and min_cost_dz == 0:
                # draw a green circle in the middle with no face
                ax3.scatter(0, 0, s=100, facecolors='none', edgecolors='green')
            else:
                ax3.arrow(0, 0, min_cost_dy, min_cost_dz, width=0.0005, fc='red', ec='red')
            plt.suptitle(f"{wm_name} - {dset_name} - {EP}")
            main_dir_str = f"[{main_axis[0].item():.3f}, {main_axis[1].item():.3f}, {main_axis[2].item():.3f}]"
            best_dir_str = f"[{main_axis[0].item()+min_cost_dx:.3f}, {main_axis[1].item()+min_cost_dy:.3f}, {main_axis[2].item()+min_cost_dz:.3f}]"
            ax0.set_title(f"dir {main_dir_str} - best {best_dir_str} - CTX {CTX} - {n_pred_steps} steps - {ITER}x{B} samples - err: {min_cost_perturb_error:.3f}")
            # how different are the best actions of k sample splits?
            K = 5
            split_size = len(errors) // K
            split_best_dxyz = []
            split_min_costs = []
            for k in range(K):
                split_dxs = dxs[k*split_size:(k+1)*split_size]
                split_dys = dys[k*split_size:(k+1)*split_size]
                split_dzs = dzs[k*split_size:(k+1)*split_size]
                split_errors = list(errors[k*split_size:(k+1)*split_size])
                # add the 0, 0, 0 sample to all splits
                if ADD_ZERO_SAMPLE:
                    split_dxs.append(0)
                    split_dys.append(0)
                    split_dzs.append(0)
                    split_errors.append(errors[zero_sample_idx])
                split_best_idx = np.argmin(split_errors)
                split_best_dxyz.append([split_dxs[split_best_idx], split_dys[split_best_idx], split_dzs[split_best_idx]])
                split_min_costs.append(split_errors[split_best_idx])
                ax1.arrow(0, 0, split_dxs[split_best_idx], split_dys[split_best_idx], width=0.0005, fc='yellow', ec='yellow', alpha=0.5)
                ax2.arrow(0, 0, split_dxs[split_best_idx], split_dzs[split_best_idx], width=0.0005, fc='yellow', ec='yellow', alpha=0.5)
                ax3.arrow(0, 0, split_dys[split_best_idx], split_dzs[split_best_idx], width=0.0005, fc='yellow', ec='yellow', alpha=0.5)
            # Add a subplot below
            first_image = context_and_future_obs[0].image # (3, H, W)
            last_image = context_and_future_obs[-1].image # (3, H, W)
            mix_image = first_image * 0.5 + last_image * 0.5
            # convert to uint8
            mix_image = (mix_image * 255).astype(np.uint8)
            # imshow
            ax0.imshow(mix_image.transpose(1, 2, 0))
            # no ticks
            ax0.set_xticks([])
            ax0.set_yticks([])
            # plt.show()
            # savefig to log folder
            png_path = os.path.join(eval_folder, f"perturberror_{dset_name.replace('/', '-')}_{EP.replace('/', '-')}_nsteps{n_pred_steps}_CAM{CAM}_CTX{CTX}.png")
            fig.savefig(png_path)

            # show best latents
            VIDEO = False
            if VIDEO:
                pred_imgs = wm.decode_latents(best_latents)
                context_and_future_decoded = wm.decode_latents(context_and_future_latents)
                mp4_filepath = os.path.join(eval_folder, f"perturberror_{dset_name.replace('/', '-')}_EP{EP.replace('/', '-')}_nsteps{n_pred_steps}_CAM{CAM}_CTX{CTX}.mp4")
                RobotObsHistory.store_side_by_side([context_and_future_decoded, pred_imgs], filepath=mp4_filepath)

            result = {
                "calibration_error_m": min_cost_perturb_error,
                "context_length": context,
                "episode": EP,
                "camera": CAM,
                "pred_length": n,
                "pred_length_in_steps": n_pred_steps,
                "dataset": dset_name,
                "wm_name": wm_name,
            }
            results.append(result)

            # log to wandb
            wandb.log({
                f"{EVAL_NAME}/metrics/calibration_error_m": min_cost_perturb_error,
                f"{EVAL_NAME}/context_length": context,
                f"{EVAL_NAME}/episode": EP,
                f"{EVAL_NAME}/camera": CAM,
                f"{EVAL_NAME}/pred_length": n,
                f"{EVAL_NAME}/pred_length_in_steps": n_pred_steps,
                f"{EVAL_NAME}/dataset": dset_name,
                f"{EVAL_NAME}/wm_name": wm_name,
                f"{EVAL_NAME}/sample_idx": len(results),
            })
            # video
            if VIDEO:
                wandb.log({
                    f"{EVAL_NAME}/videos/{dset_name.replace('/', '-')}_EP{EP.replace('/', '-')}_nsteps{n_pred_steps}_CAM{CAM}_CTX{CTX}": wandb.Video(mp4_filepath, fps=pred_imgs.freq, format="mp4"),
                })
            # png figure
            wandb.log({
                f"{EVAL_NAME}/figures/{dset_name.replace('/', '-')}_EP{EP.replace('/', '-')}_nsteps{n_pred_steps}_CAM{CAM}_CTX{CTX}": wandb.Image(png_path),
            })

            if DEV:
                # are any dx, dy, dz equal to 0?
                dxyz = np.array([dxs, dys, dzs]).T # (S, 3)
                dxyz_norm = np.linalg.norm(dxyz, axis=1) # (S,)
                print("Smallest sampled perturbation: ", np.min(dxyz_norm))

            if DEV:
                plt.show()
                # raise ValueError

    # log as wandb table
    table = wandb.Table(columns=[key for key in results[0].keys()], data=[list(result.values()) for result in results])
    wandb.log({
        f"{EVAL_NAME}/table": table,
    })
    wandb.log({
        f"{EVAL_NAME}/metrics/camera_dependent_calibration_error": wandb.plot.bar(
            table, "camera", "calibration_error_m", title="Calibration error by camera")
    })

    # mean calibration error
    mean_calibration_error = np.mean([result["calibration_error_m"] for result in results])
    print(f"Mean calibration error: {mean_calibration_error:.3f} m")
    # log to wandb
    wandb.log({
        f"{EVAL_NAME}/metrics/mean_calibration_error_m": mean_calibration_error,
    })

